# Rare-Event Inference -- DS Interview Reference (R)

Sibling to the *Statistical Inference* notebook. Where that notebook covers means
and proportions, this one covers **what changes when events are rare** -- counts
instead of means, exposure instead of sample size, exact intervals instead of
normal approximations. All examples are framed around fleet safety data (contacts,
disengagements, injuries per million miles) because that is the interview context.

## Quick Index

| Goal | Section |
| :--- | :--- |
| Why Poisson? Exposure thinking, rate vs. proportion | SS1 |
| Exact CIs for a single rate | SS2 |
| Comparing two rates (before vs. after, fleet A vs. B) | SS3 |
| Power and required exposure ("how many miles?") | SS4 |
| Adjust for confounders / ODD shift -- Poisson GLM | SS5 |
| Zero events so far -- what can we say? | SS6 |
| Overdispersion -- when variance >> mean | SS7 |
| Base rates & detector alarms (Bayes application) | SS8 |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` | Cleaning & QC |
| `DM_Intermediate` | Reshaping & combining |
| `DM_Advanced` | Feature engineering |
| `Reference_BaseR` | Base R syntax |
| `Reference_dplyr` | dplyr / tidyverse syntax |
| `Reference_Sampling` | Sampling methods & bootstrap |
| `Reference_Inference` | Hypothesis testing, A/B design |
| `Reference_RareEvents` **(this note)** | Rare-event rates, exact inference, Poisson GLM |

---

> **Interview note:** Rare-event inference is the *highest-differentiation* topic
> in a Waymo-style DS screen. Most candidates answer means/proportions questions
> fluently but freeze on "2 contact events in 10 million miles -- give me a CI."
> The three things to demonstrate: (1) use exposure (miles), not sample size (trips);
> (2) reach for `poisson.test()` not a z-interval; (3) know *why* the normal
> approximation fails at small counts.

---

## When to Reach for Each Section

| Interview signal | Section |
| :--- | :--- |
| "Events per million miles", "rate not proportion" | SS1 |
| "Give me a CI for this rate", "2 events in 10M miles" | SS2 |
| "Did the rate change after the update?", "compare two fleets" | SS3 |
| "How many miles to prove we're safer than humans?" | SS4 |
| "The ODD / city mix changed too", "adjust for confounders" | SS5 |
| "Zero events so far -- what can we claim?" | SS6 |
| "Some vehicles have way more events than others" | SS7 |
| "Anomaly alarm fired -- is it real?" | SS8 |

---
## Shared Helpers (used throughout)

In [ ]:
# ── Reusable bootstrap CI (mirrors Reference_Sampling SS5 exactly) ─────────────
bootstrap_ci <- function(data, stat_fn, B = 2000, confidence = 0.95, seed = 42) {
  set.seed(seed)
  n          <- length(data)
  observed   <- stat_fn(data)
  boot_stats <- replicate(B, stat_fn(sample(data, n, replace = TRUE)))
  alpha      <- 1 - confidence
  lo         <- quantile(boot_stats, alpha / 2)
  hi         <- quantile(boot_stats, 1 - alpha / 2)
  list(observed = observed, lo = lo, hi = hi, boot_stats = boot_stats)
}

# ── Rate helpers ──────────────────────────────────────────────────────────────
rate_per_million <- function(events, miles) sum(events) / sum(miles) * 1e6

# Quick self-check
cat("rate_per_million(c(2,3), c(5e6,5e6)) =",
    rate_per_million(c(2, 3), c(5e6, 5e6)),
    "(expect 0.5)\n")

---
# Part 1 -- The Poisson Model

---
## SS1 -- Poisson Model & Exposure Thinking

**Intuition:** A proportion answers "what fraction of trials succeeded?" A **rate**
answers "how many events per unit of exposure?". When events are rare and exposure
varies across units, rates are the right frame: two vehicles each driving 1 million
miles with 1 contact each have the *same* safety rate even though vehicle A drove
city routes and vehicle B drove highways.

The Poisson distribution models counts of rare, independent events over a fixed
exposure. Its single parameter λ is the **expected count**; for a rate r over
exposure T, λ = r × T.

> **Interview prompts:**
> *(a) Why use Poisson rather than Binomial for fleet contact events?*
> *(b) Our two cities have very different mileage. How do you compute the fleet rate?*

In [ ]:
library(dplyr)
library(ggplot2)
set.seed(42)

# ── Simulate a fleet of 200 vehicles with unequal exposure ────────────────────
true_rate  <- 2 / 1e6          # 2 contact events per million miles (human baseline ~2/M)
n_vehicles <- 200

fleet <- tibble(
  vehicle_id = 1:n_vehicles,
  city       = sample(c("SF", "Phoenix"), n_vehicles, replace = TRUE, prob = c(0.6, 0.4)),
  miles      = pmax(ifelse(city == "SF",
                 rnorm(n_vehicles, 40000, 8000),     # SF vehicles drive less (dense city)
                 rnorm(n_vehicles, 70000, 12000)),   # Phoenix vehicles drive more
               5000)
) |>
  mutate(events = rpois(n_vehicles, lambda = true_rate * miles))

cat("Fleet summary:\n")
print(fleet |> group_by(city) |>
  summarise(vehicles = n(), total_miles = sum(miles), total_events = sum(events),
            .groups = "drop"))

# ── Rule 1: rate = sum(events) / sum(miles), NOT mean(per_vehicle_rate) ────────
per_vehicle_rate <- fleet$events / fleet$miles * 1e6
wrong_rate <- mean(per_vehicle_rate)          # Simpson's-paradox trap
correct_rate <- rate_per_million(fleet$events, fleet$miles)

cat("\n-- Simpson's paradox / unequal exposure trap --\n")
cat(sprintf("  Wrong  (mean of per-vehicle rates) : %.4f per million miles\n", wrong_rate))
cat(sprintf("  Correct (sum/sum)                  : %.4f per million miles\n", correct_rate))
cat(sprintf("  True rate                          : %.4f per million miles\n", true_rate * 1e6))
cat("\n  The 'wrong' approach over-weights low-mileage vehicles (numerically noisy)\n")
cat("  Phoenix vehicles drive ~1.75x more miles -- pooling without weighting distorts the rate.\n")

# ── Rule 2: check Poisson assumptions ────────────────────────────────────────
cat("\n-- Poisson goodness-of-fit: mean vs. variance (should be ~equal) --\n")
cat(sprintf("  mean(events)  = %.3f\n", mean(fleet$events)))
cat(sprintf("  var(events)   = %.3f\n", var(fleet$events)))
cat("  (Large var >> mean signals overdispersion -> see SS7)\n")

**Poisson vs. Binomial -- when to use which:**

| | Binomial | Poisson |
| :--- | :--- | :--- |
| Frame | n trials, each succeeds/fails | Events arrive over continuous exposure |
| Parameter | n, p | λ = rate × exposure |
| Exposure varies? | Awkward (n changes per unit) | Natural (offset term) |
| Events per trial | At most 1 | Unlimited |
| Rare-event limit | Bin(n, p) → Poisson(np) as n→∞, p→0 | Exact |
| Fleet safety use | Inappropriate | Correct |

**Common mistakes:**
- Computing `mean(events / miles)` instead of `sum(events) / sum(miles)` -- low-mileage
  vehicles have noisy per-vehicle rates that dominate the average; always pool numerator and
  denominator separately.
- Assuming equal exposure across vehicles and using a plain proportion test -- this is the
  setup for Simpson's paradox when cities / ODDs have different mileage distributions.
- Forgetting that λ = rate × T: if you halve the observation window, expected counts halve
  even if the true rate is unchanged.

---
## SS2 -- Exact CIs for a Single Rate

Three methods, same structure as Inference SS2. **Lead with `poisson.test()` in
an interview** -- it is exact, requires no assumptions about count size, and is one
line. Show the normal approximation only to explain *why* it fails.

> **Interview prompt:**
> *"We observed 2 contact events in 10 million miles. Give me a 95% CI for the rate."*

In [ ]:
# ── Observed data (the classic Waymo-style screen question) ─────────────────
x     <- 2          # events
T_mi  <- 10e6       # miles of exposure

# ── Method 1: Normal approximation (and its failure) ─────────────────────────
rate_hat  <- x / T_mi * 1e6                # estimated rate per million miles
se_approx <- sqrt(x) / T_mi * 1e6         # SE for Poisson count: sqrt(lambda) / T
ci_norm   <- rate_hat + c(-1, 1) * 1.96 * se_approx

cat("-- Method 1: Normal approximation --\n")
cat(sprintf("  Rate estimate : %.4f per million miles\n", rate_hat))
cat(sprintf("  95%% CI        : (%.4f, %.4f)\n", ci_norm[1], ci_norm[2]))
cat(sprintf("  Lower bound   : %.4f  <-- can go negative at small x!\n", ci_norm[1]))

# ── Method 2: Exact Poisson CI via poisson.test() ────────────────────────────
pt_result <- poisson.test(x, T = T_mi / 1e6)   # T in same units as rate (per million miles)
ci_exact  <- pt_result$conf.int * 1             # already per-million-miles scaled

cat("\n-- Method 2: Exact (poisson.test) -- PREFERRED --\n")
cat(sprintf("  95%% CI        : (%.4f, %.4f)\n", ci_exact[1], ci_exact[2]))
cat(sprintf("  p-value vs H0: rate=0  : %.4f\n", pt_result$p.value))

# ── Method 3: Bootstrap over vehicles ────────────────────────────────────────
set.seed(42)
n_v     <- 200
fleet_s <- tibble::tibble(
  miles  = pmax(rnorm(n_v, 50000, 10000), 5000),
  events = rpois(n_v, lambda = (x / T_mi) * miles)
)

boot_rate_fn <- function(idx) {
  rate_per_million(fleet_s$events[idx], fleet_s$miles[idx])
}
boot_stats <- replicate(2000, {
  idx <- sample(n_v, n_v, replace = TRUE)
  boot_rate_fn(idx)
})
ci_boot <- quantile(boot_stats, c(0.025, 0.975))

cat("\n-- Method 3: Bootstrap over vehicles --\n")
cat(sprintf("  95%% CI        : (%.4f, %.4f)\n", ci_boot[1], ci_boot[2]))
cat("  (Useful when fleet heterogeneity, not just count noise, drives uncertainty)\n")

# ── Side-by-side comparison ───────────────────────────────────────────────────
cat("\n-- Summary --\n")
cat(sprintf("  %-22s  lower   upper\n", "Method"))
cat(sprintf("  %-22s  %.4f  %.4f   <- can go negative\n", "Normal approx", ci_norm[1], ci_norm[2]))
cat(sprintf("  %-22s  %.4f  %.4f   <- use this\n",        "Exact (poisson.test)", ci_exact[1], ci_exact[2]))
cat(sprintf("  %-22s  %.4f  %.4f   <- for fleet heterogeneity\n", "Bootstrap (vehicle)", ci_boot[1], ci_boot[2]))

**Method comparison:**

| Method | Requires | Fails when | Use in interview |
| :--- | :--- | :--- | :--- |
| Normal approximation | x ≥ ~30 | x is small (CI goes negative) | Only to show the failure mode |
| `poisson.test(x, T)` | Independence | Rate varies across exposure | Yes -- one line, exact |
| Bootstrap over vehicles | Vehicle-level data | Only have aggregate count | When fleet heterogeneity matters |

**Common mistakes:**
- Using `prop.test` or a z-interval -- these assume binomial, not Poisson; the denominator
  is exposure (miles), not a trial count.
- Passing raw miles as `T` without matching units to the rate scale -- if your rate is "per
  million miles", pass `T = miles / 1e6` so `poisson.test` returns CIs on the same scale.
- Treating the CI as symmetric -- Poisson CIs are right-skewed at small counts; the exact
  CI lower bound is not `estimate - margin` and the upper bound is not `estimate + margin`.

---
## SS3 -- Comparing Two Rates

The "did the software update reduce the contact rate?" question.
`poisson.test()` handles two-sample rate comparison exactly -- pass vectors
of counts and exposures.

> **Interview prompts:**
> *(a) Contact rate dropped from 2.1 to 1.4 per million miles after a software update.*
> *Is this statistically significant?*
> *(b) What confounders would you worry about? (Bridge to SS5.)*

In [ ]:
# ── Observed before / after data ─────────────────────────────────────────────
x_before <- 21L;   T_before <- 10e6   # 21 events in 10M miles (2.1 / M)
x_after  <- 14L;   T_after  <- 10e6   # 14 events in 10M miles (1.4 / M)

rate_before <- x_before / T_before * 1e6
rate_after  <- x_after  / T_after  * 1e6
cat(sprintf("Rate before: %.2f  |  Rate after: %.2f  |  Observed reduction: %.1f%%\n",
            rate_before, rate_after,
            (rate_before - rate_after) / rate_before * 100))

# ── Method 1: Exact two-sample Poisson test ───────────────────────────────────
pt2 <- poisson.test(c(x_after, x_before),
                    T = c(T_after, T_before) / 1e6,
                    alternative = "less")   # test: after rate < before rate

cat("\n-- Exact two-sample Poisson test --\n")
cat(sprintf("  Rate ratio (after/before)   : %.3f\n", pt2$estimate))
cat(sprintf("  95%% CI on ratio             : (%.3f, %.3f)\n",
            pt2$conf.int[1], pt2$conf.int[2]))
cat(sprintf("  p-value (one-sided, less)   : %.4f\n", pt2$p.value))
cat(sprintf("  Significant at alpha=0.05?  : %s\n", ifelse(pt2$p.value < 0.05, "YES", "NO")))

# ── Method 2: Conditional simulation (from-scratch, no package) ───────────────
# Key fact: the exact two-sample Poisson test CONDITIONS on the total count.
# Under H0 (equal rates), given total = x_after + x_before, the after-count is
# Binomial(total, T_after / (T_after + T_before)). Simulate that directly:
set.seed(42)
total  <- x_before + x_after
p_null <- T_after / (T_after + T_before)   # = 0.5 for equal exposure

B         <- 50000
sim_after <- rbinom(B, size = total, prob = p_null)
p_sim     <- mean(sim_after <= x_after)    # one-sided: as extreme or lower

cat("\n-- Conditional simulation test (from scratch) --\n")
cat(sprintf("  Observed after-count        : %d of %d total events\n", x_after, total))
cat(sprintf("  p-value (simulation, B=%d): %.4f\n", B, p_sim))
cat("  (Agrees with poisson.test because both condition on the total --\n")
cat("   explaining WHY you condition is a strong interview signal)\n")

# ── Effect size: rate difference and ratio with CIs ──────────────────────────
rate_diff <- rate_before - rate_after
cat(sprintf("\n-- Effect size --\n"))
cat(sprintf("  Absolute reduction: %.2f events per million miles\n", rate_diff))
cat(sprintf("  Relative reduction: %.1f%%\n", rate_diff / rate_before * 100))
cat("\n  Confounders to name in the interview:\n")
cat("    - ODD shift: did the update deploy to easier routes first?\n")
cat("    - Seasonal / weather change between periods\n")
cat("    - Mileage composition (city vs highway mix changed)\n")
cat("    -> If any of these, stratify by confounder or move to GLM (SS5)\n")

**Common mistakes:**
- Comparing raw event counts without normalizing by exposure -- 21 vs. 14 events
  means nothing if the "before" period had twice the mileage.
- Using a two-sample t-test on counts -- counts are non-negative and skewed; the
  Poisson model is correct here.
- Forgetting the direction of `alternative` -- `"less"` tests whether the *first*
  argument's rate is less than the second's; double-check your argument order.
- Stopping at significance without naming confounders -- Waymo interviewers
  specifically probe whether the ODD or route mix changed between periods.

---
## SS4 -- Power and Required Exposure

"How many miles do we need to drive to prove we're safer than a human driver?"
This is Waymo's most quantitative interview question. Answer it with simulation
(the pattern from Inference SS8) adapted for Poisson counts.

> **Interview prompt:**
> *The human baseline is ~2 contact events per million miles. Our system is at 1.4.*
> *How many miles of driving gives us 80% power to detect this improvement at α=0.05?*

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
rate_human <- 2.0     # per million miles (human baseline, publicly reported)
rate_av    <- 1.4     # per million miles (AV target -- the SS3 post-update rate)
alpha      <- 0.05
target_pwr <- 0.80
B          <- 5000    # simulations per mileage candidate

# ── Simulation power function ──────────────────────────────────────────────────
power_at_miles <- function(miles, r_av = rate_av, r_human = rate_human,
                           alpha = 0.05, B = 5000, seed = 42) {
  set.seed(seed)
  T_mil <- miles / 1e6    # convert to per-million scale used by poisson.test
  rejections <- replicate(B, {
    x_av    <- rpois(1, r_av    * T_mil)
    x_human <- rpois(1, r_human * T_mil)
    # one-sided: test H0: rate_av >= rate_human
    p <- poisson.test(c(x_av, x_human),
                      T = c(T_mil, T_mil),
                      alternative = "less")$p.value
    p < alpha
  })
  mean(rejections)
}

# ── Power curve across mileage candidates ─────────────────────────────────────
miles_grid <- c(1e6, 2e6, 5e6, 10e6, 20e6, 50e6, 100e6)
cat("-- Power curve: rate_av=1.4 vs rate_human=2.0 per million miles --\n")
cat(sprintf("  %-18s  %8s\n", "Miles driven", "Power"))
power_vals <- numeric(length(miles_grid))
for (i in seq_along(miles_grid)) {
  power_vals[i] <- power_at_miles(miles_grid[i])
  cat(sprintf("  %-18s  %7.1f%%\n",
              format(miles_grid[i], big.mark = ",", scientific = FALSE),
              power_vals[i] * 100))
}

# ── Find the required miles for 80% power ─────────────────────────────────────
# Interpolate between grid points
above <- which(power_vals >= target_pwr)[1]
if (is.na(above)) {
  cat("\n  80% power not reached in grid -- increase miles_grid upper bound\n")
} else {
  cat(sprintf("\n  ~%.0fM miles needed for 80%% power (30%% relative improvement)\n",
              miles_grid[above] / 1e6))
}

# ── One-sample variant: human baseline treated as a KNOWN constant ────────────
# In practice the human rate comes from billions of human-driven miles, so it is
# effectively known. Testing against a fixed r halves the noise -> less mileage.
power_one_sample <- function(miles, r_av = rate_av, r_human = rate_human,
                             alpha = 0.05, B = 5000, seed = 42) {
  set.seed(seed)
  T_mil <- miles / 1e6
  mean(replicate(B, {
    x_av <- rpois(1, r_av * T_mil)
    poisson.test(x_av, T = T_mil, r = r_human, alternative = "less")$p.value < alpha
  }))
}
cat("\n-- One-sample power (baseline known) vs. two-sample --\n")
for (m in c(10e6, 20e6, 50e6)) {
  cat(sprintf("  %5.0fM miles : one-sample %.1f%%  vs  two-sample %.1f%%\n",
              m / 1e6, power_one_sample(m) * 100, power_at_miles(m, B = 3000) * 100))
}
cat("  Treating the baseline as known cuts required mileage roughly in half.\n")

# ── The Kalra & Paddock insight: detecting a SMALL improvement is very hard ────
cat("\n-- Kalra & Paddock insight --\n")
cat("   Detecting only a 20% improvement (2.0 -> 1.6 per M miles) is far harder.\n")
cat("   Power at 100M miles for 20% improvement:\n")
p_small <- power_at_miles(100e6, r_av = 1.6, B = 3000)
cat(sprintf("   Power = %.1f%% at 100M miles  (human-driven comparison feasibility)\n",
            p_small * 100))
cat("   This is why Waymo uses counterfactual / simulation benchmarking\n")
cat("   rather than waiting for head-to-head statistical significance.\n")

**Key takeaways for the interview:**
- At a 30% relative improvement (2.0 → 1.4 per M miles), the two-sample design needs
  roughly **100M miles** for 80% power; treating the human baseline as a known constant
  (one-sample test) roughly halves that -- still tens of millions of miles.
- At a 20% improvement, the required mileage enters the **billions** -- this is the
  Kalra & Paddock (2016) result, and the reason Waymo uses simulation-based and
  counterfactual benchmarking rather than direct statistical comparison.
- Always state your assumptions: equal exposure in both arms, constant rate, no
  confounders. Each relaxation requires more miles or a richer model.

**Common mistakes:**
- Using `power.t.test` for count/rate data -- that function assumes normally distributed
  outcomes; at the event counts relevant here, the Poisson simulation is necessary.
- Forgetting to state the minimum detectable effect (MDE) alongside required miles --
  power is always conditional on an assumed effect size.
- Quoting a single miles number without a power curve -- always show how power grows
  with mileage; interviewers follow up on what happens at half the exposure.

---
# Part 2 -- Adjusting for Confounders

---
## SS5 -- Poisson GLM with Exposure Offset

When confounders (city, weather, ODD) differ between periods or fleets, a raw
rate comparison is biased. The Poisson GLM with `offset(log(miles))` is the
standard fix: it models the log-rate as a linear function of covariates, so
each coefficient is a **rate ratio** after adjustment.

> **Interview prompts:**
> *(a) The ODD changed between your before/after periods. How do you handle it?*
> *(b) Interpret the output: what does exp(coef) mean here?*

In [ ]:
library(dplyr)
set.seed(42)

# ── Simulate confounded fleet data ────────────────────────────────────────────
# True rate: 20% lower post-update, but Phoenix (hot/dry) has 40% higher base rate
n         <- 400
true_base <- 2.0 / 1e6   # base rate (SF, pre-update)

# NOTE: rows here are multi-year vehicle aggregates (~1M miles each), not single
# vehicles -- with rare events you need substantial pooled exposure per row for
# the GLM to have any information. At 50k miles/row you'd have ~0.1 expected
# events per row and the model could not recover the effects.
fleet_glm <- tibble(
  vehicle_id = 1:n,
  period     = sample(c("pre", "post"), n, replace = TRUE),
  city       = sample(c("SF", "Phoenix"), n, replace = TRUE, prob = c(0.5, 0.5)),
  miles      = pmax(rnorm(n, 1e6, 2e5), 1e5)
) |> mutate(
  rate_true = true_base *
    ifelse(period == "post", 0.80, 1.00) *   # 20% update benefit
    ifelse(city == "Phoenix", 1.40, 1.00),   # Phoenix harder ODD
  events    = rpois(n, lambda = rate_true * miles)
)

cat("-- Confounded naive comparison --\n")
naive <- fleet_glm |>
  group_by(period) |>
  summarise(events = sum(events), miles = sum(miles),
            rate_per_M = events / miles * 1e6, .groups = "drop")
print(naive)
cat("  Naive ratio (post/pre):", round(naive$rate_per_M[naive$period=="post"] /
                                       naive$rate_per_M[naive$period=="pre"], 3),
    "  <- confounded by city mix\n")

# ── Poisson GLM with offset ───────────────────────────────────────────────────
fleet_glm <- fleet_glm |>
  mutate(period = relevel(factor(period), ref = "pre"),
         city   = relevel(factor(city),   ref = "SF"))

fit <- glm(events ~ period + city + offset(log(miles)),
           data = fleet_glm, family = poisson)

cat("\n-- Poisson GLM with offset(log(miles)) --\n")
cat("  Coefficients (log rate ratios):\n")
print(round(coef(fit), 4))

cat("\n  Rate ratios = exp(coef)  [interpret as multiplicative effects on rate]:\n")
rr <- exp(cbind(RateRatio = coef(fit), confint(fit)))
print(round(rr, 3))

cat("\n  Interpretation:\n")
cat(sprintf("  periodpost  : post-update rate is %.1f%% of pre-update rate (adj. for city)\n",
            exp(coef(fit)["periodpost"]) * 100))
cat(sprintf("  cityPhoenix : Phoenix rate is %.1f%% of SF rate (adj. for period)\n",
            exp(coef(fit)["cityPhoenix"]) * 100))
cat("  True effects: 80% (update), 140% (Phoenix) -- both inside the 95% CIs.\n")

**`poisson.test` vs. Poisson GLM:**

| | `poisson.test` | Poisson `glm` with offset |
| :--- | :--- | :--- |
| Covariates | None | Any number |
| Inference | Exact | Asymptotic (Wald / profile) |
| Exposure | Single T per group | `offset(log(miles))` per row |
| Output | Rate ratio + exact CI | Rate ratios via `exp(coef)` + `confint` |
| When to use | Clean two-group, no confounders | Confounders, interactions, multiple groups |

**Common mistakes:**
- Omitting `offset(log(miles))` -- without it the model predicts log-count, not
  log-rate; coefficients lose their rate-ratio interpretation.
- Passing `log(miles)` inside a `mutate` and adding it as a regular predictor --
  this forces its coefficient to be estimated (not fixed at 1), which is wrong;
  always use `offset()`.
- Exponentiating before checking model fit -- run `summary(fit)` and check
  residual deviance / df ≈ 1; if it's >> 1, you have overdispersion (see SS7)
  and the SEs are underestimated.

---
## SS6 -- Zero Events: The Rule of Three

When x = 0, `poisson.test(0, T)` gives the exact upper bound. The **rule of
three** is the one-line interview answer: the 95% upper bound on the rate is
approximately 3 / T.

> **Interview prompt:**
> *"We've driven 5 million miles with zero injuries. What's the most we can claim?"*

In [ ]:
# ── Exact upper bound via poisson.test ───────────────────────────────────────
T_miles <- 5e6      # miles with zero events

# CAREFUL: poisson.test's conf.int is TWO-sided. The rule of three is a
# ONE-sided 95% bound, so we must ask poisson.test for conf.level = 0.90
# (two-sided 90% <=> one-sided 95% upper).
pt_zero     <- poisson.test(0, T = T_miles / 1e6, conf.level = 0.90)
upper_exact <- pt_zero$conf.int[2]

# Or directly: exact one-sided upper bound is qgamma(0.95, shape = x + 1) / T
upper_gamma <- qgamma(0.95, shape = 0 + 1) / (T_miles / 1e6)

cat("-- Zero events in 5M miles --\n")
cat(sprintf("  Exact one-sided 95%% upper bound     : %.4f per million miles\n", upper_exact))
cat(sprintf("  Same via qgamma(0.95, x+1)/T        : %.4f\n", upper_gamma))

# ── Rule of three (approximation) ────────────────────────────────────────────
upper_rule3 <- 3 / (T_miles / 1e6)
cat(sprintf("  Rule of three (3 / T)               : %.4f per million miles\n", upper_rule3))
cat("  Rule of three matches the exact ONE-sided bound (2.996 ≈ 3).\n")
cat("  (A naive poisson.test() call gives 0.7378 -- that's the TWO-sided\n")
cat("   upper bound, i.e. a 97.5% one-sided bound. Know the difference.)\n")

# ── Why "3"? Derive it ────────────────────────────────────────────────────────
# P(X=0 | lambda) = exp(-lambda). Set = 0.05 and solve for lambda:
# exp(-lambda) = 0.05  =>  lambda = -log(0.05) = 2.996 ≈ 3
lambda_upper <- -log(0.05)
cat(sprintf("\n-- Why '3'? --\n"))
cat(sprintf("  P(X=0 | lambda) = exp(-lambda) = 0.05  =>  lambda = %.4f ≈ 3\n", lambda_upper))
cat("  So 'rate × T < 3' ensures P(zero events) >= 5% even under the worst-case rate.\n")

# ── How does the upper bound tighten with more miles? ────────────────────────
cat("\n-- One-sided 95% upper bound vs. miles driven (zero events) --\n")
cat(sprintf("  %-15s  %s\n", "Miles driven", "Upper bound (per M miles)"))
for (m in c(1e6, 5e6, 10e6, 20e6, 50e6)) {
  ub <- qgamma(0.95, shape = 1) / (m / 1e6)
  cat(sprintf("  %-15s  %.4f\n",
              format(m, big.mark = ",", scientific = FALSE), ub))
}
cat("  Upper bound scales as 3/T -- double the miles, halve the worst-case rate.\n")

**Common mistakes:**
- Claiming "we proved zero risk" -- zero events only bounds the rate from above;
  it does not prove the rate is zero.
- Comparing the rule of three to `poisson.test`'s default `conf.int` -- that
  interval is two-sided, so its upper limit is a 97.5% one-sided bound
  (-log(0.025) ≈ 3.69, not 3). Use `conf.level = 0.90` or `qgamma(0.95, x+1)/T`.
- Using the rule for non-zero counts -- the approximation only holds at x = 0;
  for x ≥ 1, use `poisson.test(x, T)`.

---
## SS7 -- Overdispersion & Negative Binomial

Poisson assumes variance = mean. Real fleets violate this: some vehicles always
operate in hard ODDs, some drivers are riskier. When variance >> mean, Poisson
SEs are underestimated and CIs are too narrow. Detect with residual deviance;
fix with quasi-Poisson or `MASS::glm.nb`.

> **Interview prompt:**
> *"Your Poisson model has residual deviance of 480 on 197 df. What does that tell you?"*

In [ ]:
library(MASS, exclude = "select")   # glm.nb; exclude= avoids masking dplyr::select
set.seed(42)

# ── Simulate overdispersed fleet (heterogeneous vehicles) ─────────────────────
# Rows are cumulative vehicle histories (~2M miles) so expected counts are large
# enough for overdispersion to be VISIBLE: NB variance = mu + mu^2/theta, and the
# mu^2/theta term is negligible when mu ~ 0.1 but dominant when mu ~ 4.
n       <- 200
miles_v <- pmax(rnorm(n, 2e6, 4e5), 2e5)

# Overdispersion: each vehicle has its own latent rate (Gamma mixing -> Neg Binomial)
theta_true  <- 2              # NB dispersion (lower = more overdispersed)
mu_v        <- 2 / 1e6 * miles_v
events_od   <- rnbinom(n, size = theta_true, mu = mu_v)

cat("-- Overdispersion check: Poisson assumption --\n")
cat(sprintf("  mean(events)  = %.3f\n", mean(events_od)))
cat(sprintf("  var(events)   = %.3f\n", var(events_od)))
cat(sprintf("  Ratio var/mean= %.2f  (should be ~1 for Poisson; >>1 = overdispersed)\n",
            var(events_od) / mean(events_od)))

# ── Fit Poisson (misspecified) vs. Quasi-Poisson vs. NB ──────────────────────
df_od <- data.frame(events = events_od, miles = miles_v)

fit_pois  <- glm(events ~ offset(log(miles)), data = df_od, family = poisson)
fit_quasi <- glm(events ~ offset(log(miles)), data = df_od, family = quasipoisson)
fit_nb    <- glm.nb(events ~ offset(log(miles)), data = df_od)

cat("\n-- Residual deviance diagnostic --\n")
cat(sprintf("  Poisson  deviance/df = %.2f  (expect ~1; >> 1 confirms overdispersion)\n",
            fit_pois$deviance / fit_pois$df.residual))

cat("\n-- SE comparison (intercept / rate coefficient) --\n")
cat(sprintf("  %-20s  SE = %.4f  (underestimated -- invalid)\n", "Poisson",
            summary(fit_pois)$coef[1, 2]))
cat(sprintf("  %-20s  SE = %.4f  (correct for overdispersion)\n", "Quasi-Poisson",
            summary(fit_quasi)$coef[1, 2]))
cat(sprintf("  %-20s  SE = %.4f  (correct; estimates theta)\n", "Neg. Binomial",
            summary(fit_nb)$coef[1, 2]))
cat(sprintf("  Estimated NB theta: %.2f  (true: %.1f)\n",
            fit_nb$theta, theta_true))
cat("\n  Quasi-Poisson and NB give similar SEs; NB is preferred when theta is of interest.\n")

**Overdispersion detection & response:**

| Signal | What it means | Fix |
| :--- | :--- | :--- |
| Residual deviance / df >> 1 | Variance >> mean (overdispersed) | Quasi-Poisson or NB |
| Residual deviance / df ≈ 1 | Poisson variance assumption holds | Keep Poisson |
| Residual deviance / df << 1 | Underdispersion (rare in safety data) | Investigate model |

**Why fleets overdisperse:** vehicles differ in ODD difficulty, driver behaviour,
and maintenance state. The Gamma-Poisson mixture (which gives the Negative Binomial)
is the natural model for this latent heterogeneity.

**Common mistakes:**
- Reporting Poisson CIs without checking deviance/df -- narrow CIs from a
  misspecified Poisson model give false confidence.
- Choosing NB over quasi-Poisson based on comfort -- both fix the SE; use NB
  when you want to report θ or make predictions; quasi-Poisson is simpler and
  sufficient for inference.
- Confusing overdispersion with outliers -- a single vehicle with 50 events
  warrants investigation before modelling, not just a distributional fix.

---
## SS8 -- Base Rates & Detector Alarms

When true events are rare, even a highly accurate detector produces mostly false
alarms -- the base-rate neglect problem. This section grounds the Bayes theorem
application in the "unreliable radar alarm" interview question.

> **Interview prompt:**
> *"An anomaly detector fires on a vehicle. The detector has 95% sensitivity and
> 98% specificity. Vehicle anomalies occur 1 in 1,000 trips. Should you intervene?"*

In [ ]:
# ── Base rate neglect: compute PPV (precision) at different priors ─────────────
ppv <- function(prevalence, sensitivity, specificity) {
  tp <- prevalence       * sensitivity
  fp <- (1 - prevalence) * (1 - specificity)
  tp / (tp + fp)
}
npv <- function(prevalence, sensitivity, specificity) {
  tn <- (1 - prevalence) * specificity
  fn <- prevalence       * (1 - sensitivity)
  tn / (tn + fn)
}

sens <- 0.95   # P(alarm | true event)
spec <- 0.98   # P(no alarm | no event)

cat("-- P(true event | alarm) at varying base rates --\n")
cat(sprintf("  Detector: sensitivity=%.0f%%, specificity=%.0f%%\n", sens*100, spec*100))
cat(sprintf("  %-20s  %8s  %8s  %8s\n", "Prevalence (base rate)", "PPV", "NPV", "FP per TP"))
for (prev in c(1/10, 1/100, 1/1000, 1/10000)) {
  p <- ppv(prev, sens, spec)
  n <- npv(prev, sens, spec)
  cat(sprintf("  %-20s  %7.1f%%  %7.1f%%  %8.1f\n",
              sprintf("1 in %s", format(1/prev, big.mark=",")),
              p*100, n*100, (1-p)/p))
}
cat("\n  Key insight: at 1/1000 prevalence, ~95% of alarms are FALSE POSITIVES.\n")

# ── Threshold tuning: precision-recall tradeoff ───────────────────────────────
cat("\n-- Threshold tuning: FP cost vs. missed-event cost --\n")
cat("  In safety contexts, the cost asymmetry drives threshold choice:\n")
cat("  - Low threshold (high sensitivity): catch more true events, more false alarms\n")
cat("    -> Appropriate when missing a real event is catastrophic (e.g., brake failure)\n")
cat("  - High threshold (high specificity): fewer false alarms, miss some real events\n")
cat("    -> Appropriate when intervention cost is high (e.g., pulling vehicle from service)\n")

# ── Simulation: show alarm distribution under H0 and H1 ──────────────────────
set.seed(42)
n_trips   <- 100000
prev      <- 1 / 1000
true_anom <- rbinom(n_trips, 1, prev)
score     <- ifelse(true_anom == 1,
                    rbeta(n_trips, 6, 2),     # anomalies score high (mean 0.75) but overlap...
                    rbeta(n_trips, 2, 6))      # ...with normals (mean 0.25) -> real tradeoff

thresholds <- seq(0.3, 0.9, by = 0.1)
cat(sprintf("\n  %-10s  %8s  %8s  %8s\n", "Threshold", "Precision", "Recall", "F1"))
for (thr in thresholds) {
  preds <- as.integer(score >= thr)
  tp    <- sum(preds == 1 & true_anom == 1)
  fp    <- sum(preds == 1 & true_anom == 0)
  fn    <- sum(preds == 0 & true_anom == 1)
  prec  <- if (tp + fp > 0) tp / (tp + fp) else NA
  rec   <- if (tp + fn > 0) tp / (tp + fn) else NA
  f1    <- if (!is.na(prec) && !is.na(rec) && prec + rec > 0)
             2 * prec * rec / (prec + rec) else NA
  cat(sprintf("  %-10.1f  %7.1f%%  %7.1f%%  %7.3f\n",
              thr, prec*100, rec*100, f1))
}

**Common mistakes:**
- Equating sensitivity with precision -- sensitivity is P(alarm | true event);
  precision (PPV) is P(true event | alarm). At low prevalence, high sensitivity
  does not imply high precision.
- Choosing a threshold without stating the cost asymmetry -- in safety, false
  negatives (missed real events) typically cost more than false positives; make
  the asymmetry explicit.
- Forgetting to update the prior -- if a second independent alarm fires on the
  same vehicle, use the posterior from the first alarm as the new prior.

---
## Decision Guide

```
Rare-event question?
│
├─ Single rate, need CI
│   ├─ x = 0              -> Rule of three: upper ≈ 3/T  (SS6)
│   └─ x >= 1             -> poisson.test(x, T)  (SS2)
│
├─ Compare two rates (before/after, two fleets)
│   ├─ No confounders     -> poisson.test(c(x1,x2), T=c(T1,T2))  (SS3)
│   └─ Confounders exist  -> Poisson GLM + offset(log(miles))  (SS5)
│
├─ How many miles for target power?
│                          -> Simulate power curve  (SS4)
│
├─ variance >> mean in fleet data
│                          -> Check deviance/df; use quasi-Poisson or glm.nb  (SS7)
│
└─ Interpreting an alarm / detector
                           -> Bayes: PPV depends on base rate  (SS8)
```

**Sanity habits for every rate answer:**
1. Always report **exposure alongside counts** (`21 events / 10M miles`, not just `21`).
2. Always state the **CI method** (`poisson.test`, bootstrap, GLM Wald).
3. Always ask **"did the denominator change too?"** before comparing rates across periods.
4. When the result is surprising, check for **ODD / city mix shift** before claiming
   the rate truly changed.

---
## Sessionization Appendix

*(Fits here because exposure computation starts with pings → trips.)*

```r
# Sessionize GPS pings into trips: a new trip starts after a gap > threshold
library(dplyr)
library(lubridate)

pings |>
  arrange(vehicle_id, timestamp) |>
  group_by(vehicle_id) |>
  mutate(
    gap_sec  = as.numeric(timestamp - lag(timestamp), units = "secs"),
    new_trip = coalesce(gap_sec > 300, TRUE),   # >5 min gap = new trip; TRUE for first row
    trip_id  = cumsum(new_trip)
  ) |>
  ungroup()
```

Then compute miles per trip (e.g., cumulative haversine) and use `trip_id`
as the unit of exposure in the rate calculations above.
